In [1]:
!pip install optuna mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import tensorflow as tf
from tensorflow import keras
import optuna
import mlflow
import numpy as np
import matplotlib.pyplot as plt

# Cargar el dataset MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalizar los datos (escala de 0 a 1) para que la red aprenda mejor
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print(f"Forma de los datos de entrenamiento: {X_train.shape}")
print(f"Forma de las etiquetas de entrenamiento: {y_train.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Forma de los datos de entrenamiento: (60000, 28, 28)
Forma de las etiquetas de entrenamiento: (60000,)


In [15]:
!pip install wandb optuna
import wandb
import optuna
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Iniciar sesión en la nube
wandb.login("API KEY")

# 2. Callback personalizado a prueba de errores de versión
class SimpleWandbCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            wandb.log(logs)

# 3. Definir la búsqueda de la red (sin regularización)
def objective(trial):
    # Iniciamos el registro en WandB para cada intento
    wandb.init(project="MNIST_Optuna_Arquitectura", reinit=True, name=f"Prueba_{trial.number}")

    n_layers = trial.suggest_int("n_layers", 1, 3)
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "rmsprop", "sgd"])
    activation = trial.suggest_categorical("activation", ["relu", "tanh", "swish"])

    model = models.Sequential()
    model.add(layers.Flatten(input_shape=(28, 28)))

    for i in range(n_layers):
        n_units = trial.suggest_int(f"n_units_l{i}", 32, 256, step=32)
        model.add(layers.Dense(units=n_units, activation=activation))

    model.add(layers.Dense(10, activation="softmax"))

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    if optimizer_name == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == "rmsprop":
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # Entrenamos usando nuestro callback personalizado
    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=5,
        batch_size=64,
        callbacks=[SimpleWandbCallback()],  # <--- Usado aquí
        verbose=0
    )

    val_accuracy = history.history["val_accuracy"][-1]
    wandb.finish() # Cerramos el registro actual

    return val_accuracy

# 4. Correr Optuna para encontrar la mejor red
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("¡Listo! La mejor arquitectura encontrada es:")
print(study.best_params)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
[I 2026-09-14 02:46:04,644] A new study created in memory with name: no-name-fd0c1bb9-7aa0-4761-b2a6-1256bd27eb4b
wandb: Currently logged in as: rosi-s-s-09 (rosi-s-s-09-buap) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▄▆▇█
loss,█▆▅▃▁
val_accuracy,▁▄▆▇█
val_loss,█▇▅▃▁
accuracy,0.55785
loss,2.11263
val_accuracy,0.59767
val_loss,2.07825


[I 2026-09-14 02:46:32,329] Trial 0 finished with value: 0.5976666808128357 and parameters: {'n_layers': 3, 'optimizer': 'sgd', 'activation': 'swish', 'n_units_l0': 256, 'n_units_l1': 224, 'n_units_l2': 128, 'lr': 0.00040959516878267144}. Best is trial 0 with value: 0.5976666808128357.


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▆▇██
loss,█▅▃▂▁
val_accuracy,▁▅▇██
val_loss,█▅▃▂▁
accuracy,0.8197
loss,0.87611
val_accuracy,0.867
val_loss,0.75079


... (22 outputs left out)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▇▇██
loss,█▂▁▁▁
val_accuracy,▁▆▅▆█
val_loss,▇▁█▆▂
accuracy,0.9718
loss,0.12776
val_accuracy,0.97433
val_loss,0.11923


[I 2026-09-14 02:50:14,156] Trial 9 finished with value: 0.9743333458900452 and parameters: {'n_layers': 3, 'optimizer': 'rmsprop', 'activation': 'swish', 'n_units_l0': 96, 'n_units_l1': 256, 'n_units_l2': 128, 'lr': 0.0056305065195117136}. Best is trial 6 with value: 0.9766666889190674.


¡Listo! La mejor arquitectura encontrada es:
{'n_layers': 1, 'optimizer': 'rmsprop', 'activation': 'swish', 'n_units_l0': 256, 'lr': 0.0004974785208527011}


In [16]:
import wandb
import tensorflow as tf
from tensorflow.keras import layers, models

# Definimos las 5 regularizaciones
regularization_experiments = {
    "Baseline_Sin_Reg": None,
    "L1": tf.keras.regularizers.L1(0.01),
    "L2": tf.keras.regularizers.L2(0.01),
    "L1_L2": tf.keras.regularizers.L1L2(l1=0.01, l2=0.01),
    "Dropout": "dropout",
    "Dropout_L1_L2": "dropout_l1_l2"
}

class SimpleWandbCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            wandb.log(logs)

for reg_name, reg_val in regularization_experiments.items():
    # Inicializamos un experimento independiente en WandB para cada regularización
    wandb.init(project="MNIST_Regularizaciones", name=f"Modelo_{reg_name}", reinit=True)

    model = models.Sequential()
    model.add(layers.Flatten(input_shape=(28, 28)))

    # Aplicamos la arquitectura óptima de Optuna (1 capa de 256 neuronas, swish)
    if reg_name == "Dropout":
        model.add(layers.Dense(units=256, activation="swish"))
        model.add(layers.Dropout(0.3))
    elif reg_name == "Dropout_L1_L2":
        model.add(layers.Dense(units=256, activation="swish", kernel_regularizer=tf.keras.regularizers.L1L2(l1=0.01, l2=0.01)))
        model.add(layers.Dropout(0.3))
    elif reg_val is not None:
        model.add(layers.Dense(units=256, activation="swish", kernel_regularizer=reg_val))
    else:
        model.add(layers.Dense(units=256, activation="swish"))

    # Capa de salida
    model.add(layers.Dense(10, activation="softmax"))

    # Compilamos usando el mejor optimizador y learning rate de Optuna
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.00049747)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    # Entrenamos por 10 épocas para observar el comportamiento frente al sobreajuste
    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=10,
        batch_size=64,
        callbacks=[SimpleWandbCallback()],
        verbose=0
    )

    wandb.finish()

print("¡Listo! Todos los experimentos de regularización se han subido a WandB.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▄▆▆▇▇████
loss,█▄▃▂▂▂▁▁▁▁
val_accuracy,▁▅▆▇▇█▇███
val_loss,█▄▃▂▂▂▁▁▁▁
accuracy,0.98904
loss,0.03926
val_accuracy,0.98033
val_loss,0.06609


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▅▆▇▇▇████
loss,█▂▁▁▁▁▁▁▁▁
val_accuracy,▁▄▆▆▇▇████
val_loss,█▅▄▃▃▂▂▂▁▁
accuracy,0.85596
loss,1.36109
val_accuracy,0.8815
val_loss,1.27267


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▄▅▅▆▆▇▇██
loss,█▃▂▂▂▁▁▁▁▁
val_accuracy,▂▃▁▆▆█▇█▆▆
val_loss,█▆▅▃▂▁▂▁▁▂
accuracy,0.9193
loss,0.39011
val_accuracy,0.93267
val_loss,0.36048


... (3 outputs left out)

accuracy,▁▅▆▇▇▇████
loss,█▄▃▂▂▂▁▁▁▁
val_accuracy,▁▄▆▇▇▇▇███
val_loss,█▅▃▂▂▁▁▁▁▁
accuracy,0.98006
loss,0.06679
val_accuracy,0.9815
val_loss,0.06645


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


accuracy,▁▄▅▆▇▇▇███
loss,█▂▂▁▁▁▁▁▁▁
val_accuracy,▁▃▅▅▇▇▇█▇█
val_loss,█▆▄▄▃▂▂▁▂▁
accuracy,0.80639
loss,1.57883
val_accuracy,0.88433
val_loss,1.40635


¡Listo! Todos los experimentos de regularización se han subido a WandB.
